# Phase F - Hoc cam xuc TU CHINH AUDIO (emotion2vec+ -> bieu cam toan mat)

**Van de cua Phase B (C4):** bieu cam tinh = bang tra theo nhan (`lech TB theo nhan`, +10.7%), vung may / mui / mat la
hang so; vector cam xuc chi giup chuyen dong +0.3 diem. Tran (TB cua chinh clip) la +63% -> phan lon sai so nam o
**cuong do / sac thai bieu cam cua tung cau**, thu ma nhan roi rac khong mang duoc.

**Cach lam:** lay **vector dac trung** (khong lay nhan) cua emotion2vec+ (dong bang) va day mot nhanh nho hoc:

```
audio -+- mel ----------------> C4 (giu nguyen) ------------------> chuyen dong ham + mieng
       |                           ^ dieu kien 7 chieu (xac suat x muc/3)
       +- emotion2vec+ -> vector 1024 -> EmoBranch (MLP ~0.5M) -+-> bieu cam tinh TOAN MAT (136)
                                     [nhan: tuy chon]            +-> 8 cam xuc + muc 1..3
```

* **Khong nhan** (tu dong hoan toan) hoac **co nhan** (nguoi dung ep cam xuc, audio quyet dinh cuong do): mot mo hinh,
  train voi *label dropout*.
* Loss: LMD bieu cam tinh (muc tieu = TB cua chinh clip) + phan loai cam xuc + muc + **loss nhat quan cam xuc**
  (bo phan loai cam xuc tren landmark, dong bang - y tuong EMOTE).
* Tang cuong: embedding cua **cua so con ngau nhien** (>= 2 s) trong clip -> giong luc chay demo (cua so 2.5 s truot).

**Tiet kiem Drive:** emotion2vec+ cho ~50 khung/giay x 1024 chieu. Embedding cua emotion2vec+ = trung binh cac khung,
nen chi luu **trung binh moi doan `CHUNK_SEC` giay** (float16) -> ~250 MB cho toan bo MEAD (luu tung khung se ~12 GB).
Model emotion2vec+ (~1 GB) tai vao o dia Colab, **khong** luu len Drive. audio.tar chep tam vao `/tmp` roi xoa.

| Cell | Viec | Thoi gian (T4) |
| :--- | :--- | :--- |
| 0 - 2 | tim Drive, cau hinh, cai dat | ~2 phut |
| 3 | trich dac trung emotion2vec+ cho 41 speaker (**tu chay tiep** neu bi ngat) | ~1 - 2 gio, chi chay 1 lan |
| 4 | nap du lieu (bieu cam tinh tung clip + dac trung) | ~3 phut |
| 5 | bo phan loai cam xuc tren landmark (cho loss + metric Emo-Acc) | < 1 phut |
| 6 | baseline tuyen tinh (ridge) + train EmoBranch (**tu resume**) | ~5 phut |
| 7 | danh gia tren 4 speaker test, so voi C4 / bang tra / emotion2vec+ -> nhan -> bang tra | ~5 phut |
| 8 | xuat `phaseF_final.pth` cho demo | |

Ket qua: `student_a2l/phaseF/` (`phaseF_evaluation.json`, `phaseF_table.md`, `phaseF_final.pth`, ...).

## Cell 0 - Mount Drive va tu tim thu muc du lieu (dung duoc voi tai khoan Google khac co shortcut)

In [1]:
# Cell 0 - Mount Drive + TU TIM thu muc du lieu (chay duoc ca khi dung tai khoan Google khac, co shortcut)
import os, sys, glob
from pathlib import Path

DRIVE_ROOT_OVERRIDE = ''      # neu tu tim sai: dien tay, vd '/content/drive/MyDrive/DAT301m/Data_student'
MEAD_ROOT_OVERRIDE  = ''      # vd '/content/drive/.shortcut-targets-by-id/1GwXP.../MEAD'

if 'google.colab' in sys.modules and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')


def _is_data_root(p):
    # thu muc Data_student that: co du lieu MEAD da xu ly hoac thu muc ket qua student_a2l
    return os.path.isdir(os.path.join(p, 'Data_emotion_mead_task_b')) or os.path.isdir(os.path.join(p, 'student_a2l'))


def _is_mead(p):
    return os.path.isdir(p) and any(os.path.isfile(os.path.join(p, d, 'audio.tar')) for d in ('M003', 'M005', 'W033', 'M032'))


def _score(p):
    # 3: co du lieu MEAD da xu ly (split.json) | 2: co thu muc MEAD da xu ly | 1: chi co thu muc ket qua student_a2l
    if os.path.isfile(os.path.join(p, 'Data_emotion_mead_task_b', 'phaseB_ready', 'split.json')):
        return 3
    if os.path.isdir(os.path.join(p, 'Data_emotion_mead_task_b')):
        return 2
    return 1 if os.path.isdir(os.path.join(p, 'student_a2l')) else 0


def find_drive_root():
    if DRIVE_ROOT_OVERRIDE:
        return Path(DRIVE_ROOT_OVERRIDE)
    base = '/content/drive'
    cands = [base + '/MyDrive/DAT301m/Data_student']
    # shortcut nam o MyDrive (ten bat ky), shortcut tro toi DAT301m hoac Data_student, thu muc chia se (Shared drives)
    for pat in ('/MyDrive/*/Data_student', '/MyDrive/*/*/Data_student', '/MyDrive/Data_student',
                '/.shortcut-targets-by-id/*/DAT301m/Data_student', '/.shortcut-targets-by-id/*/Data_student',
                '/.shortcut-targets-by-id/*/*/Data_student', '/.shortcut-targets-by-id/*/*/*/Data_student',
                '/Shareddrives/*/DAT301m/Data_student', '/Shareddrives/*/Data_student', '/MyDrive/*'):
        cands += sorted(glob.glob(base + pat))
    seen, scored = set(), []
    for c in cands:
        r = os.path.realpath(c)
        if r in seen or not os.path.isdir(c):
            continue
        seen.add(r)
        sc = _score(c)
        if sc:
            scored.append((sc, c))
    for sc, c in sorted(scored, key=lambda x: -x[0]):
        print('  ung vien: %-90s diem %d' % (c, sc))
    best = max(scored, key=lambda x: x[0]) if scored else None     # max giu thu tu uu tien khi bang diem
    return Path(best[1]) if best else None


def find_mead_root():
    if MEAD_ROOT_OVERRIDE:
        return Path(MEAD_ROOT_OVERRIDE)
    base = '/content/drive'
    for pat in ('/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD', '/.shortcut-targets-by-id/*/MEAD',
                '/MyDrive/MEAD', '/MyDrive/*/MEAD', '/MyDrive/*/*/MEAD', '/Shareddrives/*/MEAD', '/Shareddrives/*/*/MEAD'):
        for c in sorted(glob.glob(base + pat)):
            if _is_mead(c):
                return Path(c)
    return None


def _writable(p):
    try:
        t = Path(p) / '.write_test'
        t.write_text('ok'); t.unlink()
        return True
    except Exception:
        return False


if 'google.colab' in sys.modules:
    DRIVE_FOUND, MEAD_FOUND = find_drive_root(), find_mead_root()
    assert DRIVE_FOUND is not None, ('Khong tim thay thu muc Data_student. Kiem tra shortcut (chuot phai thu muc DAT301m '
                                     'duoc chia se > Organize > Add shortcut > My Drive) hoac dien DRIVE_ROOT_OVERRIDE.')
    print('Data_student :', DRIVE_FOUND, '| ghi duoc:', _writable(DRIVE_FOUND))
    if _score(str(DRIVE_FOUND)) < 3:
        print('!!! Thu muc nay KHONG co Data_emotion_mead_task_b/phaseB_ready/split.json -> chua thay shortcut toi thu muc '
              'du lieu that. Them shortcut vao My Drive hoac dien DRIVE_ROOT_OVERRIDE.')
    print('MEAD goc     :', MEAD_FOUND if MEAD_FOUND else 'KHONG THAY (chi can cho cac cell doc audio MEAD goc)')
    if not _writable(DRIVE_FOUND):
        print('!!! Tai khoan nay chi co quyen XEM thu muc chia se -> khong luu duoc ket qua. '
              'Nho chu thu muc cap quyen "Nguoi chinh sua" (Editor) cho tai khoan nay.')

Mounted at /content/drive
  ung vien: /content/drive/MyDrive/DAT301m/Data_student                                                diem 3
Data_student : /content/drive/MyDrive/DAT301m/Data_student | ghi duoc: True
MEAD goc     : /content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD


## Cell 1 - Cau hinh

In [2]:
import os, sys, json, time, math, shutil, subprocess, tarfile, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/hien087585/EXPRESSIVE-2D-CHARACTER-ANIMATION.git'
REPO_DIR = Path('/content/MakeItTalk')
DRIVE    = Path('/content/drive/MyDrive/DAT301m/Data_student')
DRIVE    = globals().get('DRIVE_FOUND') or DRIVE          # Cell 0 tu tim (tai khoan khac / shortcut)
MEAD_SRC = Path('/content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD')
MEAD_SRC = globals().get('MEAD_FOUND') or MEAD_SRC
LIB_DIR  = Path('/content')
TMP      = Path('/tmp/phaseF')
_T = os.environ.get('PHASEF_TEST_ROOT')                # chi dung khi chay thu ngoai Colab
if _T:
    REPO_DIR, DRIVE, MEAD_SRC, LIB_DIR, TMP = (Path(_T) / 'repo', Path(_T) / 'drive', Path(_T) / 'mead', Path(_T),
                                               Path(_T) / 'tmp')
READY    = DRIVE / 'Data_emotion_mead_task_b' / 'phaseB_ready'
STAGE    = DRIVE / 'Data_emotion_mead_task_b' / 'stage_b_mead_out'      # zip landmark (de ghep audio <-> clip)
E2V_DIR  = DRIVE / 'Data_emotion_mead_task_b' / 'phaseF_e2v'           # dac trung emotion2vec+ (~250 MB)
PHASEB   = DRIVE / 'student_a2l' / 'phaseB'
C4_CKPT  = PHASEB / 'C4_motion_emo_e20_lr3e4' / 'phaseB_best.pth'     # mo hinh chuyen dong (giu nguyen)
OFF_PATH = PHASEB / 'label_offsets_train.npz'                          # bang tra theo nhan (Phase B Cell 7 da tao)
OUT      = DRIVE / 'student_a2l' / 'phaseF'

E2V_SIZE  = 'large'      # 'large' (1024 chieu, tot nhat) | 'base' (768 chieu, nhanh hon)
CHUNK_SEC = 1.0          # do dai moi doan luu tru (giay). 0 = chi luu 1 vector / clip (~55 MB, mat tang cuong cua so con)
RUN_EXTRACT = True       # Cell 3: tu bo qua speaker da trich xong
EMO_NAMES = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
REGIONS   = {'ham': range(0, 17), 'may': range(17, 27), 'mui': range(27, 36), 'mat': range(36, 48), 'mieng': range(48, 68)}

CFG = dict(
    window=18, audio_lead=8,            # giong Phase B
    hidden=256, drop=0.3, feat_drop=0.1,
    epochs=60, batch_size=256, lr=1e-3, weight_decay=1e-2, warmup_epochs=2,
    label_dropout=0.5,                  # xac suat AN nhan khi train -> mo hinh phai tu doan tu audio
    crop_p=0.5, crop_min_sec=2.0,       # xac suat dung cua so con ngau nhien (>= 2 s) thay vi ca clip
    lambda_static=1.0, lambda_cls=0.2, lambda_lvl=0.1, lambda_cons=0.1,
    label_smooth=0.1, seed=0, auto_resume=True,
    min_steps_per_epoch=50,             # du lieu it thi lap lai nhieu lan trong 1 epoch
)
if os.environ.get('PHASEF_SMOKE'):
    CFG.update(epochs=4)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE, '| E2V_SIZE =', E2V_SIZE, '| CHUNK_SEC =', CHUNK_SEC)
if DEVICE.type != 'cuda':
    print('[canh bao] Khong co GPU: Cell 3 (trich emotion2vec+) se rat cham. Runtime > Change runtime type > T4 GPU.')

device = cpu | E2V_SIZE = large | CHUNK_SEC = 1.0
[canh bao] Khong co GPU: Cell 3 (trich emotion2vec+) se rat cham. Runtime > Change runtime type > T4 GPU.


## Cell 2 - Cai dat, lay repo, ghi thu vien, kiem tra file

In [3]:
LIB_SRC = r'''"""Phase F - hoc cam xuc TU CHINH AUDIO bang dac trung emotion2vec+ (dong bang).

* E2VFeat: chay emotion2vec+ -> dac trung tung khung (~50 khung/giay, 1024 chieu ban large) + xac suat 8 nhan.
  Utterance embedding cua emotion2vec+ = TRUNG BINH cac khung (xem funasr/models/emotion2vec/model.py), nen chi can luu
  trung binh theo tung doan CHUNK giay + so khung cua doan: ghep lai duoc embedding cua bat ky cua so nao (tiet kiem
  luu tru ~50 lan so voi luu tung khung).
* EmoBranch: MLP nho [embedding emotion2vec+ | nhan (tuy chon)] -> (a) bieu cam tinh TOAN MAT (136), (b) logits 8 cam xuc,
  (c) muc 1..3. Vector dieu kien cho mo hinh chuyen dong C4 = xac suat 7 cam xuc x muc / 3 (dung dinh dang one-hot x muc / 3
  ma C4 da hoc) -> khong can train lai C4.
* LandmarkEmoClassifier: phan loai cam xuc tu bieu cam tinh (136) -> loss nhat quan cam xuc (kieu EMOTE) va metric Emo-Acc.
"""
import numpy as np

EMO_NAMES = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
E2V_MAP = {'angry': 'angry', 'disgusted': 'disgusted', 'fearful': 'fear', 'happy': 'happy', 'neutral': 'neutral',
           'sad': 'sad', 'surprised': 'surprised', 'other': 'neutral', 'unknown': 'neutral'}


# ============================================================================= emotion2vec+ -> dac trung
def _e2v_label(lab):
    return E2V_MAP.get(str(lab).split('/')[-1].strip().lower())


class E2VFeat:
    """emotion2vec+ (FunASR), lay dac trung TUNG KHUNG + xac suat. hub 'hf' loi thi thu 'ms' (ModelScope)."""

    def __init__(self, size='large', device='cuda', hub='hf'):
        from funasr import AutoModel
        err = None
        for h in ([hub] + [x for x in ('hf', 'ms') if x != hub]):
            name = ('emotion2vec/emotion2vec_plus_%s' if h == 'hf' else 'iic/emotion2vec_plus_%s') % size
            try:
                self.m = AutoModel(model=name, hub=h, device=device, disable_update=True, disable_pbar=True,
                                   disable_log=True)
                break
            except Exception as e:
                err = e
        else:
            raise RuntimeError('Khong tai duoc emotion2vec+: %r' % err)
        self.size = size
        self.head = self._get_head()

    def _get_head(self):
        """Lop phan loai cua emotion2vec+ (Linear tren trung binh khung) -> tinh xac suat cho cua so bat ky."""
        try:
            mdl = self.m.model
            W = mdl.proj.weight.detach().float().cpu().numpy(); b = mdl.proj.bias.detach().float().cpu().numpy()
            kw = self.m.kwargs
            labels = list(getattr(kw.get('tokenizer'), 'token_list', None) or kw.get('token_list'))
            assert len(labels) == W.shape[0]
            return dict(W=W, b=b, labels=np.array([str(x) for x in labels]))
        except Exception as e:                                            # khong lay duoc -> van chay, chi mat tien ich
            print('[canh bao] khong lay duoc lop phan loai emotion2vec+:', repr(e)[:200])
            return None

    def frames(self, y16k):
        """audio float32 16 kHz -> (dac trung (T, D) float32, xac suat (8,) theo EMO_NAMES)."""
        r = self.m.generate(np.asarray(y16k, np.float32), granularity='frame', extract_embedding=True,
                            disable_pbar=True)[0]
        f = np.asarray(r['feats'], np.float32)
        if f.ndim == 1:
            f = f[None]
        return f, scores_to8(r['labels'], r['scores'])


def scores_to8(labels, scores):
    p = np.zeros(8, np.float32)
    for lab, sc in zip(labels, scores):
        k = _e2v_label(lab)
        if k is not None:
            p[EMO_NAMES.index(k)] += float(sc)
    return p / max(p.sum(), 1e-8)


def head_probs(head, emb):
    """emb (N, D) trung binh khung -> xac suat (N, 8) bang lop phan loai goc cua emotion2vec+."""
    emb = np.atleast_2d(np.asarray(emb, np.float32))
    lg = emb @ head['W'].T + head['b']
    keep = np.array([not str(l).startswith('unuse') for l in head['labels']])
    lg[:, ~keep] = -np.inf
    lg = lg - lg.max(1, keepdims=True); p = np.exp(lg); p /= p.sum(1, keepdims=True)
    out = np.zeros((len(emb), 8), np.float32)
    for j, lab in enumerate(head['labels']):
        k = _e2v_label(lab)
        if keep[j] and k is not None:
            out[:, EMO_NAMES.index(k)] += p[:, j]
    return out / np.maximum(out.sum(1, keepdims=True), 1e-8)


def chunk_means(feats, dur_sec, chunk_sec):
    """dac trung khung (T, D) -> (trung binh tung doan chunk_sec giay (K, D), so khung moi doan (K,)).
    chunk_sec <= 0: chi 1 doan = ca clip."""
    T = len(feats)
    if chunk_sec <= 0 or dur_sec <= chunk_sec:
        return feats.mean(0, keepdims=True), np.array([T], np.int32)
    fps = T / max(dur_sec, 1e-6)
    edges = np.unique(np.clip(np.round(np.arange(0, dur_sec + 1e-9, chunk_sec) * fps).astype(int), 0, T))
    if edges[-1] != T:
        edges = np.append(edges, T)
    if T - edges[-2] < 0.4 * chunk_sec * fps and len(edges) > 2:        # doan cuoi qua ngan -> gop vao doan truoc
        edges = np.delete(edges, -2)
    M = np.stack([feats[a:b].mean(0) for a, b in zip(edges[:-1], edges[1:])])
    return M, np.diff(edges).astype(np.int32)


def pool(chunks, counts, a=0, b=None):
    """trung binh co trong so cua cac doan [a, b) = embedding cua cua so do."""
    c = counts[a:b].astype(np.float32)
    return (chunks[a:b].astype(np.float32) * c[:, None]).sum(0) / max(c.sum(), 1.0)


def window_embeddings(feats, dur_sec, win=2.5, hop=0.5):
    """Cho audio dai (demo): dac trung khung -> (tam cua so giay (N,), embedding (N, D)). Cua so win giay, buoc hop."""
    T = len(feats); fps = T / max(dur_sec, 1e-6)
    n, h = max(1, int(round(win * fps))), max(1, int(round(hop * fps)))
    st = list(range(0, max(T - n, 0) + 1, h)) or [0]
    if st[-1] != max(T - n, 0):
        st.append(max(T - n, 0))
    cs = np.concatenate([np.zeros((1, feats.shape[1])), np.cumsum(feats, 0, dtype=np.float64)])
    E = np.stack([(cs[min(s + n, T)] - cs[s]) / (min(s + n, T) - s) for s in st]).astype(np.float32)
    return (np.array(st) + min(n, T) / 2) / fps, E


# ============================================================================= mo hinh
def build_emo_branch(in_dim=1024, label_dim=7, hidden=256, out_dim=136, drop=0.3, feat_drop=0.1):
    import torch
    import torch.nn as nn

    class EmoBranch(nn.Module):
        """[emb emotion2vec+ (D) | nhan 7 chieu x muc/3 | co nhan?] -> (tinh 136, logits 8, muc 1..3)."""

        def __init__(self):
            super().__init__()
            self.cfg = dict(in_dim=in_dim, label_dim=label_dim, hidden=hidden, out_dim=out_dim, drop=drop,
                            feat_drop=feat_drop)
            self.norm = nn.LayerNorm(in_dim)
            self.fdrop = nn.Dropout(feat_drop)
            self.body = nn.Sequential(nn.Linear(in_dim + label_dim + 1, hidden), nn.GELU(), nn.Dropout(drop),
                                      nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop))
            self.static = nn.Linear(hidden, out_dim)
            self.lin = nn.Linear(in_dim, out_dim, bias=False)     # duong tat tuyen tinh (bao gom duoc ca ridge)
            self.cls = nn.Linear(hidden, 8)
            self.lvl = nn.Linear(hidden, 1)
            for p in (self.static.weight, self.static.bias, self.lin.weight):
                nn.init.zeros_(p)

        def forward(self, emb, label_vec=None, has_label=None):
            # label_vec (B, 7) = one-hot x muc/3 (neutral = 0); has_label (B,) 1 = co nhan, 0 = chi dung audio.
            # Can co has_label rieng vi neutral co nhan cung la vector 0.
            x = self.fdrop(self.norm(emb))
            if label_vec is None:
                lab = x.new_zeros(len(x), self.cfg['label_dim'] + 1)
            else:
                has = (x.new_ones(len(x)) if has_label is None else has_label.to(x.dtype)).reshape(-1, 1)
                lab = torch.cat([label_vec.to(x.dtype) * has, has], 1)
            h = self.body(torch.cat([x, lab], 1))
            return self.static(h) + self.lin(x), self.cls(h), 1.0 + 2.0 * torch.sigmoid(self.lvl(h)[:, 0])

        @staticmethod
        def cond_vector(logits, lvl):
            """-> vector 7 chieu cho mo hinh chuyen dong C4 (xac suat cam xuc x muc / 3; neutral = 0)."""
            p = torch.softmax(logits.float(), -1)
            return p[:, 1:] * (lvl.float()[:, None] / 3.0)

    return EmoBranch()


def build_lm_classifier(in_dim=136, hidden=128, drop=0.3):
    import torch.nn as nn
    m = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, hidden), nn.GELU(),
                      nn.Dropout(drop), nn.Linear(hidden, 8))
    m.cfg = dict(in_dim=in_dim, hidden=hidden, drop=drop)
    return m


def load_emo_branch(path, device='cpu'):
    """Nap phaseF_final.pth -> (EmoBranch, checkpoint dict: chua 'e2v_head', 'chunk_sec', 'lm_scale', ...)."""
    import torch
    ck = torch.load(str(path), map_location='cpu', weights_only=False)
    m = build_emo_branch(**ck['branch_cfg'])
    m.load_state_dict(ck['branch_state'])
    return m.to(device).eval(), ck
'''
(LIB_DIR / 'emo_audio_lib.py').write_text(LIB_SRC)
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import importlib, emo_audio_lib as EA
importlib.reload(EA)

if IN_COLAB:
    if not (REPO_DIR / 'src' / 'models' / 'model_audio2landmark.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    try:
        import funasr  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'funasr', 'modelscope'], check=False)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src.models.model_audio2landmark import StudentAudio2Landmark  # noqa: E402

MEAN_STD_PATH = REPO_DIR / 'src' / 'dataset' / 'utils' / 'MEAN_STD_AUTOVC_RETRAIN_MEL_AU.txt'
for p in (READY / 'split.json', C4_CKPT, OFF_PATH, MEAN_STD_PATH):
    assert p.exists(), 'Thieu file: %s' % p
SPLIT = json.loads((READY / 'split.json').read_text())
ALL_SP = SPLIT['train'] + SPLIT['val'] + SPLIT['test']
OUT.mkdir(parents=True, exist_ok=True); E2V_DIR.mkdir(parents=True, exist_ok=True)
zo = np.load(OFF_PATH); OFF = {(int(e), int(l)): v.astype(np.float32) for e, l, v in zip(zo['emo'], zo['lvl'], zo['off'])}
print('train %d | val %s | test %s | bang tra %d nhom' % (len(SPLIT['train']), SPLIT['val'], SPLIT['test'], len(OFF)))
print('MEAD goc:', MEAD_SRC, '| ton tai:', MEAD_SRC.exists())

train 33 | val ['M003', 'M005', 'W033', 'W036'] | test ['M032', 'M037', 'W018', 'W019'] | bang tra 22 nhom
MEAD goc: /content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD | ton tai: True


## Cell 3 - Trich dac trung emotion2vec+ cho moi clip MEAD (chi chay 1 lan)

Moi speaker: chep `audio.tar` vao `/tmp`, giai nen, ghep audio <-> clip **theo thu tu trong nhom (cam xuc, muc)** (giong
luc tien xu ly), cat audio dung bang doan mel cua clip, chay emotion2vec+ -> trung binh moi `CHUNK_SEC` giay.
Luu `phaseF_e2v/<speaker>.npz` ngay sau moi speaker -> **bi ngat thi chay lai cell, no lam tiep**.

In [4]:
def audio_groups(sp, tar_local):
    # ten clip (stem) -> ten file trong audio.tar, ghep theo thu tu trong nhom (cam xuc, muc) nhu mead_preprocess.py
    zf = zipfile.ZipFile(STAGE / sp / ('%s_landmark_package.zip' % sp))
    lm = {}
    for n in sorted(i.filename for i in zf.infolist() if '/clips/' in i.filename and '__front__' in i.filename):
        q = Path(n).stem.split('__'); lm.setdefault((q[2], q[3][1:]), []).append((q[4], Path(n).stem))
    with tarfile.open(tar_local) as tf:
        names = [m.name for m in tf.getmembers() if m.name.endswith('.m4a')]
    au = {}
    for nm in names:
        p = Path(nm); au.setdefault((p.parts[-3], p.parts[-2].replace('level_', '')), []).append(nm)
    out = {}
    for k, items in lm.items():
        a = sorted(au.get(k, []), key=lambda s: Path(s).stem)
        if len(a) == len(items):
            for (sid, stem), an in zip(sorted(items), a):
                out[stem] = an
    return out


def decode16k(path):
    r = subprocess.run(['ffmpeg', '-loglevel', 'error', '-i', str(path), '-f', 'f32le', '-ac', '1', '-ar', '16000', '-'],
                       capture_output=True)
    return np.frombuffer(r.stdout, np.float32).copy() if r.returncode == 0 else np.zeros(0, np.float32)


def speaker_done(sp):
    f = E2V_DIR / (sp + '.npz')
    if not f.exists():
        return False
    try:
        z = np.load(f)
        return float(z['chunk_sec']) == CHUNK_SEC and str(z['e2v_size']) == E2V_SIZE
    except Exception:
        return False


if RUN_EXTRACT:
    todo = [sp for sp in ALL_SP if not speaker_done(sp)]
    print('Can trich: %d / %d speaker' % (len(todo), len(ALL_SP)))
    if todo:
        if os.environ.get('PHASEF_FAKE_E2V'):           # chi dung khi chay thu ngoai Colab
            E2V = __import__('fake_e2v').FakeE2V()
        else:
            E2V = EA.E2VFeat(size=E2V_SIZE, device=DEVICE.type)
        if E2V.head is not None and not (E2V_DIR / 'e2v_head.npz').exists():
            np.savez(E2V_DIR / 'e2v_head.npz', **E2V.head)
    from concurrent.futures import ThreadPoolExecutor
    for si, sp in enumerate(todo, 1):
        t0 = time.time()
        z = np.load(READY / (sp + '.npz'))
        names, off = [str(x) for x in z['names']], z['offsets']
        tmp = TMP / sp; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
        shutil.copy(MEAD_SRC / sp / 'audio.tar', tmp / 'audio.tar')
        amap = audio_groups(sp, tmp / 'audio.tar')
        need = sorted({amap[n] for n in names if n in amap}); need_set = set(need)
        with tarfile.open(tmp / 'audio.tar') as tf:
            tf.extractall(tmp / 'a', members=[m for m in tf.getmembers() if m.name in need_set], filter='data')
        (tmp / 'audio.tar').unlink()
        with ThreadPoolExecutor(4) as ex:
            wavs = dict(zip(need, ex.map(lambda nm: decode16k(tmp / 'a' / nm), need)))
        CH, CNT, COFF, P8, HAS = [], [], [0], [], []
        for i, n in enumerate(names):
            N = int(off[i + 1] - off[i])
            y = wavs.get(amap.get(n, ''), np.zeros(0, np.float32))[:int(round(N / 62.5 * 16000))]
            if len(y) < 1600:                       # thieu / hong audio -> danh dau, Cell 4 bo clip nay
                ch, cnt, p8, ok = np.zeros((1, CH[0].shape[1] if CH else 1), np.float32), np.zeros(1, np.int32), np.zeros(8, np.float32), 0
            else:
                fr, p8 = E2V.frames(y)
                ch, cnt = EA.chunk_means(fr, len(y) / 16000.0, CHUNK_SEC); ok = 1
            CH.append(ch); CNT.append(cnt); COFF.append(COFF[-1] + len(ch)); P8.append(p8); HAS.append(ok)
        D = max(c.shape[1] for c in CH)
        CH = [c if c.shape[1] == D else np.zeros((len(c), D), np.float32) for c in CH]
        np.savez(E2V_DIR / (sp + '.npz'), chunks=np.concatenate(CH).astype(np.float16), counts=np.concatenate(CNT).astype(np.int16),
                 coff=np.array(COFF, np.int32), probs8=np.stack(P8).astype(np.float16), has=np.array(HAS, np.int8),
                 names=np.array(names), chunk_sec=np.float32(CHUNK_SEC), e2v_size=np.array(E2V_SIZE))
        shutil.rmtree(tmp, ignore_errors=True)
        print('[%2d/%d] %s | %d clip | thieu audio %d | %.1f MB | %.0f s'
              % (si, len(todo), sp, len(names), len(names) - sum(HAS), (E2V_DIR / (sp + '.npz')).stat().st_size / 1e6, time.time() - t0))
tot = sum(f.stat().st_size for f in E2V_DIR.glob('*.npz')) / 1e6
print('Xong. %d / %d speaker co dac trung | tong %.0f MB tren Drive' % (sum(speaker_done(s) for s in ALL_SP), len(ALL_SP), tot))

Can trich: 0 / 41 speaker
Xong. 41 / 41 speaker co dac trung | tong 221 MB tren Drive


## Cell 4 - Nap du lieu: bieu cam tinh tung clip (muc tieu) + dac trung emotion2vec+

Muc tieu **bieu cam tinh** = trung binh landmark goc (`lm_res_raw`, do lech so voi mat trung tinh cua speaker) tren
clip, dung dung doan ma Phase B danh gia (`[0, N - audio_lead)`). Day chinh la dong `tran` cua Phase B.

In [5]:
W, LEAD = CFG['window'], CFG['audio_lead']


def emo_vec(e, l):
    v = np.zeros(7, np.float32)
    if e > 0:
        v[e - 1] = l / 3.0
    return v


def load_clips(speakers, name):
    t0 = time.time()
    Y, EMO, LVL, SPK, NAME, CH, CNT, COFF, P8 = [], [], [], [], [], [], [], [0], []
    for sp in speakers:
        # QUAN TRONG: doc moi mang 1 lan / speaker. Viet z['x'][i] trong vong lap se giai nen lai CA mang moi lan
        # (rat cham), va lat cat f['chunks'][a:b] giu song ca mang -> tran RAM.
        with np.load(READY / (sp + '.npz')) as z:
            raw, off, emo_, lvl_, nm_ = z['lm_res_raw'], z['offsets'], z['emo'], z['lvl'], [str(x) for x in z['names']]
        with np.load(E2V_DIR / (sp + '.npz')) as f:
            chunks, counts, coff, has, probs8, fnames = f['chunks'], f['counts'], f['coff'], f['has'], f['probs8'], list(f['names'])
        assert fnames == nm_, 'Thu tu clip lech o %s -> xoa %s.npz va chay lai Cell 3' % (sp, sp)
        for i in range(len(off) - 1):
            a, b = int(off[i]), int(off[i + 1])
            if (b - a) - LEAD < W or not has[i]:
                continue
            Y.append(raw[a:b - LEAD].astype(np.float32).mean(0))
            EMO.append(int(emo_[i])); LVL.append(int(lvl_[i])); SPK.append(sp); NAME.append(nm_[i])
            c0, c1 = int(coff[i]), int(coff[i + 1])
            CH.append(chunks[c0:c1].copy()); CNT.append(counts[c0:c1].copy()); COFF.append(COFF[-1] + c1 - c0)
            P8.append(probs8[i].astype(np.float32))
        del raw, chunks
    D = dict(name=name, Y=np.stack(Y), emo=np.array(EMO), lvl=np.array(LVL), sp=np.array(SPK), names=NAME,
             chunks=np.concatenate(CH), counts=np.concatenate(CNT).astype(np.float32), coff=np.array(COFF), probs8=np.stack(P8))
    D['lab'] = np.stack([emo_vec(e, l) for e, l in zip(D['emo'], D['lvl'])])
    D['full'] = np.stack([EA.pool(D['chunks'], D['counts'], D['coff'][i], D['coff'][i + 1]) for i in range(len(Y))])
    print('%-5s | %2d speaker | %5d clip | %.1f MB dac trung | %.0f s' % (name, len(speakers), len(Y), D['chunks'].nbytes / 1e6, time.time() - t0))
    return D


DTR, DVA, DTE = load_clips(SPLIT['train'], 'train'), load_clips(SPLIT['val'], 'val'), load_clips(SPLIT['test'], 'test')
E_DIM = DTR['full'].shape[1]
ZERO_LMD = float(np.linalg.norm(DTR['Y'].reshape(-1, 68, 2), axis=2).mean())    # sai so khi doan 0 -> chuan hoa loss
print('chieu dac trung = %d | LMD bieu cam tinh khi doan 0 (train) = %.4f' % (E_DIM, ZERO_LMD))


def static_lmd(P, Y):
    return np.linalg.norm((P - Y).reshape(len(Y), 68, 2), axis=2)       # (N, 68)

train | 33 speaker | 21721 clip | 174.6 MB dac trung | 129 s
val   |  4 speaker |  2592 clip | 19.4 MB dac trung | 15 s
test  |  4 speaker |  2652 clip | 23.1 MB dac trung | 16 s
chieu dac trung = 1024 | LMD bieu cam tinh khi doan 0 (train) = 0.0438


## Cell 5 - Bo phan loai cam xuc tren landmark (bieu cam tinh -> 8 cam xuc)

Hai bo, train tren bieu cam tinh **that** cua speaker train, chon theo val:
* `CLF_LOSS` (dong bang) -> **loss nhat quan cam xuc** khi train EmoBranch;
* `CLF_EVAL` (kien truc / seed khac) -> metric **Emo-Acc** (khong dung chung bo voi loss de khong tu cham diem cho minh).

In [6]:
def train_lm_clf(seed, hidden, epochs=40):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    m = EA.build_lm_classifier(hidden=hidden).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-3, weight_decay=1e-3)
    X, Yc = torch.tensor(DTR['Y'] / ZERO_LMD), torch.tensor(DTR['emo'])
    Xv = torch.tensor(DVA['Y'] / ZERO_LMD).to(DEVICE); Yv = DVA['emo']
    cw = torch.tensor(len(Yc) / (8 * np.bincount(DTR['emo'], minlength=8).clip(1)), dtype=torch.float32).to(DEVICE)
    best, state = -1, None
    for ep in range(epochs):
        m.train(); perm = rng.permutation(len(X))
        for i in range(0, len(X), 256):
            j = perm[i:i + 256]
            x = (X[j] + 0.05 * torch.randn_like(X[j])).to(DEVICE)
            loss = F.cross_entropy(m(x), Yc[j].to(DEVICE), weight=cw)
            opt.zero_grad(); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            pv = m(Xv).argmax(1).cpu().numpy()
        uar = np.mean([(pv[Yv == k] == k).mean() for k in np.unique(Yv)])
        if uar > best:
            best, state = uar, {k: v.detach().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(state); m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m, float(best)


CLF_LOSS, uar_l = train_lm_clf(0, 128)
CLF_EVAL, uar_e = train_lm_clf(1, 256)
print('Bo phan loai tren landmark THAT: val UAR %.3f (cho loss) | %.3f (cho Emo-Acc) | doan bua 0.125' % (uar_l, uar_e))

Bo phan loai tren landmark THAT: val UAR 0.684 (cho loss) | 0.686 (cho Emo-Acc) | doan bua 0.125


## Cell 6 - Baseline tuyen tinh (ridge) + train EmoBranch

* **Ridge**: hoi quy tuyen tinh embedding ca clip -> bieu cam tinh (alpha chon theo val). Moc "chi can tuyen tinh".
* **EmoBranch**: moi buoc lay embedding cua ca clip (50%) hoac cua so con ngau nhien >= 2 s (50%), an nhan voi xac suat
  `label_dropout`. Chon `best` theo **val LMD bieu cam tinh khi KHONG co nhan** (muc tieu chinh: tu doan tu audio).
  Moi epoch luu `phaseF_last.pth` -> bi ngat thi chay lai cell, tu train tiep.

In [7]:
# ---------------- ridge (tuyen tinh) ----------------
mu_x, sd_x = DTR['full'].mean(0), DTR['full'].std(0) + 1e-6
Xtr = (DTR['full'] - mu_x) / sd_x; Xva = (DVA['full'] - mu_x) / sd_x
Ym = DTR['Y'].mean(0)
U, S_, Vt = np.linalg.svd(Xtr, full_matrices=False); UtY = U.T @ (DTR['Y'] - Ym)
best_r = None
for alpha in [1, 10, 100, 1e3, 1e4, 1e5]:
    Wr = Vt.T @ ((S_ / (S_ ** 2 + alpha))[:, None] * UtY)
    v = static_lmd(Xva @ Wr + Ym, DVA['Y']).mean()
    if best_r is None or v < best_r[0]:
        best_r = (v, alpha, Wr)
RIDGE = dict(alpha=best_r[1], W=best_r[2], mu=mu_x, sd=sd_x, ym=Ym)
ridge_pred = lambda E: ((E - RIDGE['mu']) / RIDGE['sd']) @ RIDGE['W'] + RIDGE['ym']
print('Ridge: alpha = %g | val LMD tinh %.4f (doan 0: %.4f)' % (best_r[1], best_r[0], static_lmd(0 * DVA['Y'], DVA['Y']).mean()))

# ---------------- EmoBranch ----------------
torch.manual_seed(CFG['seed']); rng = np.random.default_rng(CFG['seed'])
branch = EA.build_emo_branch(in_dim=E_DIM, hidden=CFG['hidden'], drop=CFG['drop'], feat_drop=CFG['feat_drop']).to(DEVICE)
print('EmoBranch: %.3f M tham so' % (sum(p.numel() for p in branch.parameters()) / 1e6))
opt = torch.optim.AdamW(branch.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
spe = max(math.ceil(len(DTR['Y']) / CFG['batch_size']), CFG['min_steps_per_epoch']); total = CFG['epochs'] * spe; warm = CFG['warmup_epochs'] * spe
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                          else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
cw = torch.tensor(len(DTR['emo']) / (8 * np.bincount(DTR['emo'], minlength=8).clip(1)), dtype=torch.float32).to(DEVICE)
CK_LAST, CK_BEST, HIST = OUT / 'phaseF_last.pth', OUT / 'phaseF_best.pth', OUT / 'phaseF_history.json'


def sample_emb(D, idx, train):
    out = np.empty((len(idx), E_DIM), np.float32)
    for r, i in enumerate(idx):
        a, b = int(D['coff'][i]), int(D['coff'][i + 1]); K = b - a
        if train and K > 1 and rng.random() < CFG['crop_p']:
            kmin = max(1, min(K, int(math.ceil(CFG['crop_min_sec'] / max(CHUNK_SEC, 1e-6)))))
            L = rng.integers(kmin, K + 1); s = rng.integers(0, K - L + 1)
            out[r] = EA.pool(D['chunks'], D['counts'], a + s, a + s + L)
        else:
            out[r] = D['full'][i]
    return out


def lmd_t(p, y):
    return ((p - y).reshape(len(p), 68, 2).pow(2).sum(-1) + 1e-8).sqrt().mean()


@torch.no_grad()
def predict_branch(D, with_label, emb=None):
    branch.eval()
    E = torch.tensor(D['full'] if emb is None else emb).to(DEVICE)
    lab = torch.tensor(D['lab']).to(DEVICE)
    has = torch.full((len(E),), 1.0 if with_label else 0.0, device=DEVICE)
    S, L, V = branch(E, lab, has)
    return S.cpu().numpy() * ZERO_LMD, torch.softmax(L, -1).cpu().numpy(), V.cpu().numpy(), branch.cond_vector(L, V).cpu().numpy()


def val_scores():
    s0, p0, _, _ = predict_branch(DVA, False); s1, _, _, _ = predict_branch(DVA, True)
    pv = p0.argmax(1); yv = DVA['emo']
    return dict(lmd_audio=float(static_lmd(s0, DVA['Y']).mean()), lmd_label=float(static_lmd(s1, DVA['Y']).mean()),
                uar=float(np.mean([(pv[yv == k] == k).mean() for k in np.unique(yv)])))


start, best, history = 0, float('inf'), []
if CFG['auto_resume'] and CK_LAST.exists():
    ck = torch.load(CK_LAST, map_location=DEVICE, weights_only=False)
    branch.load_state_dict(ck['branch_state']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
    rng = np.random.default_rng(CFG['seed'] + start)
    print('>> RESUME tu epoch %d (best val LMD tinh, khong nhan = %.4f)' % (start, best))
elif CFG['auto_resume'] and CK_BEST.exists():          # da train xong truoc do (Cell 8 xoa last.pth) -> khong train lai
    ck = torch.load(CK_BEST, map_location=DEVICE, weights_only=False)
    start, best, history = CFG['epochs'], ck['best'], ck['history']
    print('>> Da co phaseF_best.pth -> bo qua train (xoa file nay neu muon train lai)')
Ytr = torch.tensor(DTR['Y'] / ZERO_LMD); LAB = torch.tensor(DTR['lab']); EMO_T = torch.tensor(DTR['emo']); LVL_T = torch.tensor(DTR['lvl'], dtype=torch.float32)
print('%-5s %-9s %-9s %-9s %-9s | %-12s %-12s %-8s' % ('epoch', 'loss', 'static', 'cls', 'cons', 'val khong nhan', 'val co nhan', 'val UAR'))
for ep in range(start, CFG['epochs']):
    branch.train(); acc_ = np.zeros(4); t0 = time.time()
    perm = np.concatenate([rng.permutation(len(DTR['Y'])) for _ in range(math.ceil(spe * CFG['batch_size'] / len(DTR['Y'])))])
    for st in range(spe):
        j = perm[st * CFG['batch_size']:(st + 1) * CFG['batch_size']]
        E = torch.tensor(sample_emb(DTR, j, True)).to(DEVICE)
        has = torch.tensor((rng.random(len(j)) >= CFG['label_dropout']).astype(np.float32)).to(DEVICE)
        y, e, l = Ytr[j].to(DEVICE), EMO_T[j].to(DEVICE), LVL_T[j].to(DEVICE)
        S, LG, LV = branch(E, LAB[j].to(DEVICE), has)
        l_static = lmd_t(S, y)
        l_cls = F.cross_entropy(LG, e, weight=cw, label_smoothing=CFG['label_smooth'])
        nz = e > 0
        l_lvl = (LV[nz] - l[nz]).abs().mean() if nz.any() else S.sum() * 0
        l_cons = F.cross_entropy(CLF_LOSS(S), e, weight=cw)
        loss = (CFG['lambda_static'] * l_static + CFG['lambda_cls'] * l_cls + CFG['lambda_lvl'] * l_lvl
                + CFG['lambda_cons'] * l_cons)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(branch.parameters(), 1.0); opt.step(); sched.step()
        acc_ += [loss.item(), l_static.item(), l_cls.item(), l_cons.item()]
    v = val_scores(); acc_ /= spe
    rec = dict(epoch=ep + 1, loss=acc_[0], static=acc_[1], cls=acc_[2], cons=acc_[3], seconds=round(time.time() - t0, 1),
               **{'val_' + k: x for k, x in v.items()})
    history.append(rec)
    is_best = v['lmd_audio'] < best
    if is_best:
        best = v['lmd_audio']
    payload = dict(branch_state=branch.state_dict(), branch_cfg=branch.cfg, opt=opt.state_dict(), sched=sched.state_dict(),
                   epoch=ep, best=best, history=history, cfg=CFG, zero_lmd=ZERO_LMD, chunk_sec=CHUNK_SEC, e2v_size=E2V_SIZE)
    torch.save(payload, CK_LAST)
    if is_best:
        shutil.copy(CK_LAST, CK_BEST)
    HIST.write_text(json.dumps(history, indent=1))
    print('%-5d %-9.4f %-9.4f %-9.4f %-9.4f | %-14.4f %-12.4f %-8.3f %s'
          % (ep + 1, acc_[0], acc_[1], acc_[2], acc_[3], v['lmd_audio'], v['lmd_label'], v['uar'], '<-- BEST' if is_best else ''))
ck = torch.load(CK_BEST, map_location=DEVICE, weights_only=False)
branch.load_state_dict(ck['branch_state']); branch.eval()
BEST_EPOCH = ck['epoch'] + 1
print('Chon epoch %d | val LMD tinh (khong nhan) %.4f | bang tra theo nhan tren val: %.4f | doan 0: %.4f'
      % (BEST_EPOCH, ck['best'], static_lmd(np.stack([OFF[(e, l)] for e, l in zip(DVA['emo'], DVA['lvl'])]), DVA['Y']).mean(),
         static_lmd(0 * DVA['Y'], DVA['Y']).mean()))

Ridge: alpha = 100000 | val LMD tinh 0.0394 (doan 0: 0.0426)
EmoBranch: 0.509 M tham so
>> Da co phaseF_best.pth -> bo qua train (xoa file nay neu muon train lai)
epoch loss      static    cls       cons      | val khong nhan val co nhan  val UAR 
Chon epoch 32 | val LMD tinh (khong nhan) 0.0392 | bang tra theo nhan tren val: 0.0382 | doan 0: 0.0426


## Cell 7 - **Danh gia** tren 4 speaker test (landmark goc, cung cach tinh voi Phase B Cell 7)

Moi ung vien = **bieu cam tinh** (theo cot 2) + **chuyen dong C4** (tru TB clip, ham duoi + mieng) voi dieu kien theo cot 3.

| Ung vien | Bieu cam tinh | Dieu kien C4 | Can nhan? |
| :--- | :--- | :--- | :--- |
| `dung yen` | 0 | khong co chuyen dong | |
| `C4 (Phase B)` | bang tra theo nhan | nhan | **co** |
| `e2v+ -> nhan -> bang tra` | bang tra theo nhan **do emotion2vec+ doan** (cach demo cu) | nhan doan | khong |
| `F-ridge` | tuyen tinh tu embedding | xac suat EmoBranch | khong |
| `F (chi audio)` | EmoBranch, khong nhan | xac suat EmoBranch | **khong** |
| `F (audio + nhan)` | EmoBranch, co nhan | nhan | co |
| `tran` | TB cua chinh clip | khong co chuyen dong | (gian lan) |

Them: **LMD bieu cam tinh** (chi so phan tinh voi TB that cua clip), **Emo-Acc** (bo phan loai `CLF_EVAL` doan cam xuc
tu bieu cam tinh du doan), **tuong quan cuong do** (do lon bieu cam du doan vs that, chi clip cam xuc), **SER**
(8 nhan, co contempt) cua EmoBranch so voi emotion2vec+.

In [8]:
ms = np.loadtxt(MEAN_STD_PATH, dtype=np.float64).reshape(2, 80)
MEL_MEAN, MEL_STD = ms[0].astype(np.float32), np.maximum(ms[1], 1e-5).astype(np.float32)
c4_meta = torch.load(str(C4_CKPT), map_location='cpu', weights_only=False)
MOTION_PTS = c4_meta.get('motion_pts') or (list(range(3, 14)) + list(range(48, 68)))
MASK = np.zeros((68, 2), np.float32); MASK[MOTION_PTS] = 1.0; MASK = MASK.reshape(136)
C4 = StudentAudio2Landmark.from_checkpoint(str(C4_CKPT), map_location='cpu').to(DEVICE).eval()

# ---- mel + landmark goc cua test, dung thu tu clip cua DTE ----
want = {(s, n) for s, n in zip(DTE['sp'], DTE['names'])}
MEL, RAW, POS = [], [], {}
pos = 0
for sp in SPLIT['test']:
    with np.load(READY / (sp + '.npz')) as z:
        mel_, raw_, off_, nm_ = z['mel'], z['lm_res_raw'], z['offsets'], [str(x) for x in z['names']]
    for i in range(len(off_) - 1):
        n = nm_[i]
        if (sp, n) not in want:
            continue
        a, b = int(off_[i]), int(off_[i + 1])
        MEL.append(((mel_[a + LEAD:b].astype(np.float32) - MEL_MEAN) / MEL_STD).astype(np.float16))
        RAW.append(raw_[a:b - LEAD].copy()); POS[(sp, n)] = (pos, b - a - LEAD); pos += b - a - LEAD
    del mel_, raw_
MEL, RAW = np.concatenate(MEL), np.concatenate(RAW).astype(np.float32)
CLIP = [POS[(s, n)] for s, n in zip(DTE['sp'], DTE['names'])]


@torch.no_grad()
def c4_motion(cond, stride=8, batch=1024):
    starts, cids = [], []
    for ci, (p, n) in enumerate(CLIP):
        st = list(range(0, n - W + 1, stride))
        if st[-1] != n - W:
            st.append(n - W)
        starts += [p + s for s in st]; cids += [ci] * len(st)
    starts, cids = np.array(starts), np.array(cids)
    acc = np.zeros((len(MEL), 136)); cnt = np.zeros(len(MEL))
    for i in range(0, len(starts), batch):
        s = starts[i:i + batch][:, None] + np.arange(W)
        out = C4(torch.from_numpy(MEL[s].astype(np.float32)).to(DEVICE),
                 torch.from_numpy(cond[cids[i:i + batch]].astype(np.float32)).to(DEVICE)).float().cpu().numpy()
        np.add.at(acc, s.ravel(), out.reshape(-1, 136)); np.add.at(cnt, s.ravel(), 1.0)
    P = (acc / np.maximum(cnt, 1)[:, None]).astype(np.float32)
    for p, n in CLIP:                           # chi giu chuyen dong (tru TB clip) tren cac diem C4 da hoc
        P[p:p + n] = (P[p:p + n] - P[p:p + n].mean(0)) * MASK
    return P


# ---- bieu cam tinh cua tung ung vien ----
def nearest_off(e, l):
    if e == 0:
        return OFF[(0, min(l for (e_, l) in OFF if e_ == 0))]
    ls = [l_ for (e_, l_) in OFF if e_ == e]
    return OFF[(e, min(ls, key=lambda x: abs(x - l)))]


e2v_lab = DTE['probs8'].argmax(1); e2v_conf = DTE['probs8'].max(1)
e2v_lvl = np.clip(np.round(1 + 2 * np.clip((e2v_conf - 0.4) / 0.5, 0, 1)), 1, 3).astype(int)   # giong probs_to_timeline
S_table = np.stack([nearest_off(e, l) for e, l in zip(DTE['emo'], DTE['lvl'])])
S_casc = np.stack([nearest_off(e, l) for e, l in zip(e2v_lab, e2v_lvl)])
S_ridge = ridge_pred(DTE['full'])
S_F, P_F, V_F, COND_F = predict_branch(DTE, False)
S_FL, _, _, _ = predict_branch(DTE, True)
COND_LAB = DTE['lab']
COND_CASC = np.stack([emo_vec(e, l) for e, l in zip(e2v_lab, e2v_lvl)])

t0 = time.time()
MOT = {'lab': c4_motion(COND_LAB), 'casc': c4_motion(COND_CASC), 'F': c4_motion(COND_F)}
print('Du doan chuyen dong C4 (3 kieu dieu kien): %.0f s' % (time.time() - t0))
CANDS = {   # ten -> (bieu cam tinh (n_clip, 136) hoac None, chuyen dong hoac None)
    'dung yen (mat trung tinh)':        (np.zeros_like(DTE['Y']), None),
    'C4 Phase B (bang tra theo NHAN)':  (S_table, MOT['lab']),
    'e2v+ -> nhan -> bang tra (cu)':    (S_casc, MOT['casc']),
    'F-ridge (tuyen tinh, chi audio)':  (S_ridge, MOT['F']),
    'F EmoBranch (CHI AUDIO)':          (S_F, MOT['F']),
    'F EmoBranch (audio + nhan)':       (S_FL, MOT['lab']),
    'tran: TB cua chinh clip':          (DTE['Y'], None),
}
NOLABEL = ['e2v+ -> nhan -> bang tra (cu)', 'F-ridge (tuyen tinh, chi audio)', 'F EmoBranch (CHI AUDIO)']

# ---- LMD toan bo (tinh + chuyen dong) so voi landmark goc, giong Phase B ----
acc, per_sp = {}, {}
for ci, (p, n) in enumerate(CLIP):
    g = RAW[p:p + n]; grp = 'neutral' if DTE['emo'][ci] == 0 else 'cam xuc'
    for name, (S, M) in CANDS.items():
        pr = S[ci][None] + (M[p:p + n] if M is not None else 0)
        e = np.linalg.norm((g - pr).reshape(n, 68, 2), axis=2)
        r = acc.setdefault((grp, name), [np.zeros(68), 0]); r[0] += e.sum(0); r[1] += n
        if grp == 'cam xuc':
            r2 = per_sp.setdefault((DTE['sp'][ci], name), [0.0, 0]); r2[0] += e.mean(1).sum(); r2[1] += n

report = dict(cfg=CFG, best_epoch=BEST_EPOCH, chunk_sec=CHUNK_SEC, e2v_size=E2V_SIZE, ridge_alpha=RIDGE['alpha'],
              groups={}, per_speaker={}, static={}, emo_acc={}, intensity={}, ser={})
for grp in ('neutral', 'cam xuc'):
    base = acc[(grp, 'dung yen (mat trung tinh)')]; base = base[0] / base[1]
    print('\n== Clip %s - LMD toan bo (tinh + chuyen dong), %% = tot hon "dung yen" ==' % grp)
    print('  %-34s %-17s %-17s %s' % ('', 'toan mat', 'mieng', 'theo vung'))
    for name in CANDS:
        m_ = acc[(grp, name)]; m_ = m_[0] / m_[1]
        gain = lambda idx: float(100 * (1 - m_[idx].mean() / base[idx].mean()))
        reg = {r: round(gain(list(i)), 1) for r, i in REGIONS.items()}
        print('  %-34s %.4f (%+5.1f%%)  %.4f (%+5.1f%%)  %s' % (name, m_.mean(), gain(slice(None)), m_[48:].mean(),
              gain(slice(48, 68)), ' '.join('%s %+.1f' % kv for kv in reg.items())))
        report['groups'].setdefault(grp, {})[name] = dict(lmd=float(m_.mean()), lmd_mouth=float(m_[48:].mean()),
                                                          gain_pct=round(gain(slice(None)), 2), regions=reg)

print('\nTung speaker test (clip cam xuc, toan mat, % tot hon "dung yen"):')
for sp in SPLIT['test']:
    b0 = per_sp[(sp, 'dung yen (mat trung tinh)')]; row = {}
    for name in list(CANDS)[1:]:
        r = per_sp[(sp, name)]; row[name] = round(float(100 * (1 - (r[0] / r[1]) / (b0[0] / b0[1]))), 1)
    report['per_speaker'][sp] = row
    print('  %s: %s' % (sp, ' | '.join('%s %+.1f' % (k.split(' (')[0], v) for k, v in row.items())))

# ---- chi phan TINH: sai so so voi TB that cua clip; Emo-Acc; tuong quan cuong do ----
emo_mask = DTE['emo'] > 0
mag = lambda S: np.linalg.norm(S.reshape(len(S), 68, 2), axis=2).mean(1)
with torch.no_grad():
    clf = lambda S: CLF_EVAL(torch.tensor(S / ZERO_LMD, dtype=torch.float32).to(DEVICE)).argmax(1).cpu().numpy()
print('\nChi BIEU CAM TINH (so voi TB that cua clip) | Emo-Acc (bo phan loai landmark doan dung cam xuc?) | cuong do:')
print('  %-34s %-20s %-20s %-10s %-10s %s' % ('', 'LMD tinh cam xuc', 'LMD tinh neutral', 'Emo-Acc', 'Emo-UAR', 'tuong quan cuong do'))
y_e = DTE['emo']
for name, (S, _) in CANDS.items():
    d = static_lmd(S, DTE['Y']).mean(1)
    z0 = static_lmd(0 * DTE['Y'], DTE['Y']).mean(1)
    gE = 100 * (1 - d[emo_mask].mean() / z0[emo_mask].mean()); gN = 100 * (1 - d[~emo_mask].mean() / z0[~emo_mask].mean())
    pc = clf(S); acc_e = float((pc == y_e).mean()); uar_e = float(np.mean([(pc[y_e == k] == k).mean() for k in np.unique(y_e)]))
    mp, mg = mag(S)[emo_mask], mag(DTE['Y'])[emo_mask]
    ic = float(np.corrcoef(mp, mg)[0, 1]) if mp.std() > 1e-9 else 0.0
    report['static'][name] = dict(lmd_emo=float(d[emo_mask].mean()), gain_emo=round(gE, 2), lmd_neu=float(d[~emo_mask].mean()), gain_neu=round(gN, 2))
    report['emo_acc'][name] = dict(acc=acc_e, uar=uar_e); report['intensity'][name] = ic
    print('  %-34s %.4f (%+6.1f%%)    %.4f (%+6.1f%%)    %-10.3f %-10.3f %.3f' % (name, d[emo_mask].mean(), gE, d[~emo_mask].mean(), gN, acc_e, uar_e, ic))

# ---- nhan cam xuc tu audio (8 nhan, co contempt) ----
uar = lambda y, p: float(np.mean([(p[y == k] == k).mean() for k in np.unique(y)]))
pF = P_F.argmax(1); k7 = y_e != 7
report['ser'] = dict(F_acc=float((pF == y_e).mean()), F_uar=uar(y_e, pF), e2v_acc=float((e2v_lab == y_e).mean()), e2v_uar=uar(y_e, e2v_lab),
                     F_uar7=uar(y_e[k7], pF[k7]), e2v_uar7=uar(y_e[k7], e2v_lab[k7]),
                     F_level_mae=float(np.abs(V_F[emo_mask] - DTE['lvl'][emo_mask]).mean()),
                     recall_F={EMO_NAMES[k]: round(float((pF[y_e == k] == k).mean()), 3) for k in np.unique(y_e)})
print('\nNhan cam xuc tu audio (test): EmoBranch acc %.3f UAR %.3f (8 nhan) | emotion2vec+ goc acc %.3f UAR %.3f'
      % (report['ser']['F_acc'], report['ser']['F_uar'], report['ser']['e2v_acc'], report['ser']['e2v_uar']))
print('  7 nhan chung (bo contempt): EmoBranch UAR %.3f | emotion2vec+ UAR %.3f | sai so muc (1..3) EmoBranch %.2f'
      % (report['ser']['F_uar7'], report['ser']['e2v_uar7'], report['ser']['F_level_mae']))
print('  recall EmoBranch:', report['ser']['recall_F'])

ge = report['groups']['cam xuc']
f, c4, cs = ge['F EmoBranch (CHI AUDIO)']['gain_pct'], ge['C4 Phase B (bang tra theo NHAN)']['gain_pct'], ge['e2v+ -> nhan -> bang tra (cu)']['gain_pct']
print('\nKET LUAN (clip cam xuc, toan mat): F chi audio %+.1f%% | C4 can nhan %+.1f%% | cach demo cu (e2v -> nhan) %+.1f%%' % (f, c4, cs))
print('  F chi audio - C4 = %+.1f diem | F chi audio - cach cu = %+.1f diem | F co nhan - C4 = %+.1f diem'
      % (f - c4, f - cs, ge['F EmoBranch (audio + nhan)']['gain_pct'] - c4))
report['summary'] = dict(F_audio=f, C4_label=c4, cascade=cs, F_label=ge['F EmoBranch (audio + nhan)']['gain_pct'],
                         oracle=ge['tran: TB cua chinh clip']['gain_pct'])

# ---- bang markdown cho bao cao ----
rows = ['| Mo hinh | Can nhan? | LMD toan mat (cam xuc) | % | LMD mieng | Vung may | Vung mat | LMD tinh | Emo-Acc | Tuong quan cuong do |',
        '| :--- | :---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
NEED = {'C4 Phase B (bang tra theo NHAN)': 'co', 'F EmoBranch (audio + nhan)': 'co', 'tran: TB cua chinh clip': '(gian lan)'}
for name in CANDS:
    g_ = ge[name]; need = 'khong' if name in NOLABEL else NEED.get(name, '-')
    rows.append('| %s | %s | %.4f | %+.1f | %.4f | %+.1f | %+.1f | %.4f | %.3f | %.3f |' % (
        name, need, g_['lmd'], g_['gain_pct'], g_['lmd_mouth'], g_['regions']['may'], g_['regions']['mat'],
        report['static'][name]['lmd_emo'], report['emo_acc'][name]['acc'], report['intensity'][name]))
(OUT / 'phaseF_table.md').write_text('\n'.join(rows) + '\n')
(OUT / 'phaseF_evaluation.json').write_text(json.dumps(report, indent=1, ensure_ascii=False, default=float))
print('\nDa luu:', OUT / 'phaseF_evaluation.json', '|', OUT / 'phaseF_table.md')

Du doan chuyen dong C4 (3 kieu dieu kien): 476 s

== Clip neutral - LMD toan bo (tinh + chuyen dong), % = tot hon "dung yen" ==
                                     toan mat          mieng             theo vung
  dung yen (mat trung tinh)          0.0221 ( +0.0%)  0.0345 ( +0.0%)  ham +0.0 may +0.0 mui +0.0 mat +0.0 mieng +0.0
  C4 Phase B (bang tra theo NHAN)    0.0207 ( +6.1%)  0.0310 (+10.1%)  ham +5.0 may -0.1 mui -0.0 mat -0.1 mieng +10.1
  e2v+ -> nhan -> bang tra (cu)      0.0214 ( +2.8%)  0.0321 ( +6.9%)  ham +3.2 may -12.2 mui -1.6 mat -2.2 mieng +6.9
  F-ridge (tuyen tinh, chi audio)    0.0254 (-15.3%)  0.0378 ( -9.6%)  ham -8.2 may -52.2 mui -26.5 mat -24.1 mieng -9.6
  F EmoBranch (CHI AUDIO)            0.0236 ( -7.1%)  0.0347 ( -0.8%)  ham -6.2 may -29.9 mui -6.7 mat -30.7 mieng -0.8
  F EmoBranch (audio + nhan)         0.0214 ( +2.9%)  0.0317 ( +8.1%)  ham +2.8 may -8.8 mui -4.7 mat -7.9 mieng +8.1
  tran: TB cua chinh clip            0.0158 (+28.2%)  0.0282 (+18.3%)  ham

## Cell 8 - Xuat `phaseF_final.pth` (cho demo) + xoa file trung gian

Gom: trong so EmoBranch, cau hinh, lop phan loai cua emotion2vec+, bang tra, duong dan C4. Xoa `phaseF_last.pth`
(chi can cho resume) de tiet kiem Drive. **Khong** xoa `phaseF_e2v/` (can neu muon train lai).

In [9]:
DELETE_LAST = True
head = dict(np.load(E2V_DIR / 'e2v_head.npz')) if (E2V_DIR / 'e2v_head.npz').exists() else None
final = dict(branch_state=branch.state_dict(), branch_cfg=branch.cfg, zero_lmd=ZERO_LMD, chunk_sec=CHUNK_SEC, e2v_size=E2V_SIZE,
             e2v_head=head, c4_ckpt=str(C4_CKPT), motion_pts=MOTION_PTS, label_offsets={'%d_%d' % k: v for k, v in OFF.items()},
             best_epoch=BEST_EPOCH, summary=report.get('summary'), emo_names=EMO_NAMES,
             usage='emb (1024) cua cua so 2.5 s -> branch(emb, label_vec, has_label) -> (tinh*zero_lmd, logits, muc)')
torch.save(final, OUT / 'phaseF_final.pth')
if DELETE_LAST and (OUT / 'phaseF_last.pth').exists():
    (OUT / 'phaseF_last.pth').unlink()
sz = lambda p: sum(f.stat().st_size for f in Path(p).rglob('*') if f.is_file()) / 1e6
print('Da luu %s (%.1f MB) | thu muc phaseF: %.1f MB | phaseF_e2v: %.0f MB'
      % (OUT / 'phaseF_final.pth', (OUT / 'phaseF_final.pth').stat().st_size / 1e6, sz(OUT), sz(E2V_DIR)))

Da luu /content/drive/MyDrive/DAT301m/Data_student/student_a2l/phaseF/phaseF_final.pth (2.1 MB) | thu muc phaseF: 8.3 MB | phaseF_e2v: 221 MB


# Phase F2 - Chi giu **neutral + 2 cam xuc tot nhat** (chon bang so lieu tren VAL)

## Cell 9 - Cham diem moi cap cam xuc tren val, chon cap tot nhat

Chay sau Cell 8 (can bien cua Cell 4 - 7 trong bo nho). Diem cua moi cap (a, b), **chi tinh tren 4 speaker val**:

| Tieu chi | Y nghia | Cach tinh |
| :--- | :--- | :--- |
| `audio` | audio phan biet duoc 3 lop {neutral, a, b} | UAR 3 lop cua EmoBranch (xac suat chi xet 3 lop) |
| `mat` | khuon mat phan biet duoc 3 lop | UAR 3 lop cua bo phan loai landmark `CLF_EVAL` tren bieu cam THAT |
| `hoc` | mo hinh tai tao duoc bieu cam tu audio | % LMD bieu cam tinh tot hon "dung yen" cua F (chi audio), TB cua a va b |
| `ro` | bieu cam du lon de nhin thay | do lon bang tra a, b so voi TB cac cam xuc (cat o 1) |

`diem = TB(audio, mat, hoc, ro)`. Muon tu chon: dat `KEEP_OVERRIDE = ('happy', 'sad')`.

In [10]:
KEEP_OVERRIDE = None          # vd ('happy', 'sad') de bo qua tu chon
from itertools import combinations


def uar_sub(y, p):
    return float(np.mean([(p[y == k] == k).mean() for k in np.unique(y)]))


SV, PV, _, _ = predict_branch(DVA, False)                    # bieu cam tinh + xac suat 8 lop, khong nhan, val
with torch.no_grad():
    PFACE = torch.softmax(CLF_EVAL(torch.tensor(DVA['Y'] / ZERO_LMD, dtype=torch.float32).to(DEVICE)), -1).cpu().numpy()
z0v = static_lmd(0 * DVA['Y'], DVA['Y']).mean(1); dFv = static_lmd(SV, DVA['Y']).mean(1)
learn = {e: float(1 - dFv[DVA['emo'] == e].mean() / z0v[DVA['emo'] == e].mean()) for e in range(1, 8)}
mag_ = {e: float(np.mean([np.linalg.norm(OFF[k].reshape(68, 2), axis=1).mean() for k in OFF if k[0] == e])) for e in range(1, 8)}
mag_mean = np.mean(list(mag_.values()))
rows_ = []
for a, b in combinations(range(1, 8), 2):
    ks = [0, a, b]; m = np.isin(DVA['emo'], ks); y = DVA['emo'][m]
    au = uar_sub(y, np.array(ks)[PV[m][:, ks].argmax(1)])
    fa = uar_sub(y, np.array(ks)[PFACE[m][:, ks].argmax(1)])
    le = 0.5 * (learn[a] + learn[b]); ro = min(1.0, 0.5 * (mag_[a] + mag_[b]) / mag_mean)
    rows_.append(dict(pair=(EMO_NAMES[a], EMO_NAMES[b]), audio=au, mat=fa, hoc=le, ro=ro, diem=float(np.mean([au, fa, le, ro]))))
rows_.sort(key=lambda r: -r['diem'])
print('%-24s %-7s %-7s %-7s %-7s %s' % ('cap (VAL)', 'audio', 'mat', 'hoc', 'ro', 'DIEM'))
for r in rows_[:10]:
    print('%-24s %-7.3f %-7.3f %-7.3f %-7.3f %.3f' % ('%s + %s' % r['pair'], r['audio'], r['mat'], r['hoc'], r['ro'], r['diem']))
print('Tung cam xuc (val): hoc', {EMO_NAMES[e]: round(v, 3) for e, v in learn.items()})
KEEP_NAMES = tuple(KEEP_OVERRIDE) if KEEP_OVERRIDE else rows_[0]['pair']
KEEP = [0] + [EMO_NAMES.index(n) for n in KEEP_NAMES]
TAG = 'keep_%s_%s' % KEEP_NAMES
print('\n-> GIU: neutral + %s + %s   (%s)' % (KEEP_NAMES[0], KEEP_NAMES[1], 'tu chon' if KEEP_OVERRIDE else 'chon theo diem val'))
(OUT / 'phaseF2_pair_selection.json').write_text(json.dumps(dict(ranking=rows_, keep=KEEP_NAMES), indent=1, default=float))

cap (VAL)                audio   mat     hoc     ro      DIEM
angry + happy            0.963   0.946   0.159   1.000   0.767
angry + surprised        0.950   0.944   0.152   1.000   0.761
angry + fear             0.958   0.922   0.131   1.000   0.753
angry + sad              0.969   0.882   0.106   1.000   0.739
disgusted + surprised    0.943   0.962   0.083   0.922   0.727
disgusted + fear         0.941   0.952   0.061   0.924   0.719
angry + disgusted        0.925   0.834   0.119   1.000   0.719
surprised + contempt     0.791   0.983   0.090   1.000   0.716
sad + surprised          0.967   0.941   0.070   0.878   0.714
fear + sad               0.968   0.950   0.048   0.880   0.711
Tung cam xuc (val): hoc {'angry': 0.189, 'disgusted': 0.049, 'fear': 0.072, 'happy': 0.13, 'sad': 0.023, 'surprised': 0.116, 'contempt': 0.064}

-> GIU: neutral + angry + happy   (chon theo diem val)


4240

## Cell 10 - Train lai EmoBranch chi tren neutral + 2 cam xuc da chon

Giong Cell 6 nhung chi dung clip thuoc 3 lop; logits cac lop khac bi chan (`-inf`) -> bai toan 3 lop.
C4 (khau hinh) giu nguyen (van hoc tren toan bo MEAD). Luu `phaseF/<TAG>/`, co resume.

In [11]:
OUT2 = OUT / TAG; OUT2.mkdir(parents=True, exist_ok=True)
KMASK = torch.full((8,), float('-inf'), device=DEVICE); KMASK[KEEP] = 0.0     # chi dung khi suy luan
KIDX = torch.tensor(KEEP, device=DEVICE)                                        # khi train: lay dung 3 cot -> CE 3 lop
REMAP = torch.full((8,), -1, dtype=torch.long, device=DEVICE); REMAP[KIDX] = torch.arange(len(KEEP), device=DEVICE)


def subset(D, keep):
    idx = np.where(np.isin(D['emo'], keep))[0]
    S = {k: (v[idx] if isinstance(v, np.ndarray) and len(v) == len(D['emo']) and k not in ('chunks', 'counts', 'coff') else v)
         for k, v in D.items() if k != 'names'}
    S['names'] = [D['names'][i] for i in idx]
    S['coff'] = D['coff']; S['chunks'] = D['chunks']; S['counts'] = D['counts']
    S['row'] = idx                                     # vi tri trong D goc (de lay chunk + cua so con)
    return S


STR, SVA = subset(DTR, KEEP), subset(DVA, KEEP)
print('train %d clip | val %d clip | lop %s' % (len(STR['emo']), len(SVA['emo']), [EMO_NAMES[k] for k in KEEP]))


def sample_emb2(D, idx, train):
    out = np.empty((len(idx), E_DIM), np.float32)
    for r, i in enumerate(idx):
        g = D['row'][i]; a, b = int(D['coff'][g]), int(D['coff'][g + 1]); K = b - a
        if train and K > 1 and rng2.random() < CFG['crop_p']:
            kmin = max(1, min(K, int(math.ceil(CFG['crop_min_sec'] / max(CHUNK_SEC, 1e-6)))))
            L = rng2.integers(kmin, K + 1); s = rng2.integers(0, K - L + 1)
            out[r] = EA.pool(D['chunks'], D['counts'], a + s, a + s + L)
        else:
            out[r] = D['full'][i]
    return out


torch.manual_seed(CFG['seed']); rng2 = np.random.default_rng(CFG['seed'])
br2 = EA.build_emo_branch(in_dim=E_DIM, hidden=CFG['hidden'], drop=CFG['drop'], feat_drop=CFG['feat_drop']).to(DEVICE)
opt2 = torch.optim.AdamW(br2.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
spe2 = max(math.ceil(len(STR['emo']) / CFG['batch_size']), CFG['min_steps_per_epoch'])
tot2, warm2 = CFG['epochs'] * spe2, CFG['warmup_epochs'] * spe2
sch2 = torch.optim.lr_scheduler.LambdaLR(opt2, lambda s: (s + 1) / warm2 if s < warm2
                                         else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm2) / max(1, tot2 - warm2)))))
cnt_ = np.bincount(STR['emo'], minlength=8).astype(np.float32)
cw2 = torch.tensor(np.where(cnt_ > 0, len(STR['emo']) / (len(KEEP) * np.maximum(cnt_, 1)), 0.0), dtype=torch.float32).to(DEVICE)


@torch.no_grad()
def pred2(D, with_label, emb=None):
    br2.eval()
    E = torch.tensor(D['full'] if emb is None else emb).to(DEVICE)
    has = torch.full((len(E),), 1.0 if with_label else 0.0, device=DEVICE)
    S, L, V = br2(E, torch.tensor(D['lab']).to(DEVICE), has)
    L = L + KMASK
    return S.cpu().numpy() * ZERO_LMD, torch.softmax(L, -1).cpu().numpy(), V.cpu().numpy(), br2.cond_vector(L, V).cpu().numpy()


def val2():
    s0, p0, _, _ = pred2(SVA, False); s1, _, _, _ = pred2(SVA, True)
    return dict(lmd_audio=float(static_lmd(s0, SVA['Y']).mean()), lmd_label=float(static_lmd(s1, SVA['Y']).mean()),
                uar=uar_sub(SVA['emo'], p0.argmax(1)))


CK2_LAST, CK2_BEST = OUT2 / 'last.pth', OUT2 / 'best.pth'
start2, best2, hist2 = 0, float('inf'), []
if CK2_LAST.exists():
    ck = torch.load(CK2_LAST, map_location=DEVICE, weights_only=False)
    br2.load_state_dict(ck['branch_state']); opt2.load_state_dict(ck['opt']); sch2.load_state_dict(ck['sched'])
    start2, best2, hist2 = ck['epoch'] + 1, ck['best'], ck['history']; rng2 = np.random.default_rng(CFG['seed'] + start2)
    print('>> RESUME tu epoch %d' % start2)
elif CK2_BEST.exists():
    start2 = CFG['epochs']; print('>> Da co %s -> bo qua train' % CK2_BEST)
Y2 = torch.tensor(STR['Y'] / ZERO_LMD); LAB2 = torch.tensor(STR['lab']); E2 = torch.tensor(STR['emo']); V2 = torch.tensor(STR['lvl'], dtype=torch.float32)
for ep in range(start2, CFG['epochs']):
    br2.train(); t0 = time.time(); acc2 = np.zeros(2)
    perm = np.concatenate([rng2.permutation(len(STR['emo'])) for _ in range(math.ceil(spe2 * CFG['batch_size'] / len(STR['emo'])))])
    for st in range(spe2):
        j = perm[st * CFG['batch_size']:(st + 1) * CFG['batch_size']]
        E = torch.tensor(sample_emb2(STR, j, True)).to(DEVICE)
        has = torch.tensor((rng2.random(len(j)) >= CFG['label_dropout']).astype(np.float32)).to(DEVICE)
        y, e, l = Y2[j].to(DEVICE), E2[j].to(DEVICE), V2[j].to(DEVICE)
        S, LG, LV = br2(E, LAB2[j].to(DEVICE), has)
        l_static = lmd_t(S, y)
        l_cls = F.cross_entropy(LG[:, KIDX], REMAP[e], weight=cw2[KIDX], label_smoothing=CFG['label_smooth'])
        nz = e > 0
        l_lvl = (LV[nz] - l[nz]).abs().mean() if nz.any() else S.sum() * 0
        l_cons = F.cross_entropy(CLF_LOSS(S)[:, KIDX], REMAP[e], weight=cw2[KIDX])
        loss = (CFG['lambda_static'] * l_static + CFG['lambda_cls'] * l_cls + CFG['lambda_lvl'] * l_lvl + CFG['lambda_cons'] * l_cons)
        opt2.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(br2.parameters(), 1.0); opt2.step(); sch2.step()
        acc2 += [loss.item(), l_static.item()]
    v = val2(); hist2.append(dict(epoch=ep + 1, loss=acc2[0] / spe2, static=acc2[1] / spe2, **{'val_' + k: x for k, x in v.items()}))
    is_best = v['lmd_audio'] < best2
    best2 = min(best2, v['lmd_audio'])
    torch.save(dict(branch_state=br2.state_dict(), branch_cfg=br2.cfg, opt=opt2.state_dict(), sched=sch2.state_dict(), epoch=ep,
                    best=best2, history=hist2, keep=KEEP), CK2_LAST)
    if is_best:
        shutil.copy(CK2_LAST, CK2_BEST)
    if (ep + 1) % 5 == 0 or is_best:
        print('%-4d loss %.4f | val tinh khong nhan %.4f | co nhan %.4f | UAR 3 lop %.3f %s'
              % (ep + 1, acc2[0] / spe2, v['lmd_audio'], v['lmd_label'], v['uar'], '<-- BEST' if is_best else ''))
ck = torch.load(CK2_BEST, map_location=DEVICE, weights_only=False); br2.load_state_dict(ck['branch_state']); br2.eval()
BEST2 = ck['epoch'] + 1
print('Chon epoch %d | val LMD tinh (khong nhan) %.4f' % (BEST2, ck['best']))

train 7117 clip | val 875 clip | lop ['neutral', 'angry', 'happy']
1    loss 1.0323 | val tinh khong nhan 0.0304 | co nhan 0.0304 | UAR 3 lop 0.963 <-- BEST
2    loss 0.9015 | val tinh khong nhan 0.0303 | co nhan 0.0303 | UAR 3 lop 0.961 <-- BEST
5    loss 0.9007 | val tinh khong nhan 0.0314 | co nhan 0.0315 | UAR 3 lop 0.959 
10   loss 0.8840 | val tinh khong nhan 0.0305 | co nhan 0.0305 | UAR 3 lop 0.963 
13   loss 0.8767 | val tinh khong nhan 0.0302 | co nhan 0.0301 | UAR 3 lop 0.947 <-- BEST
15   loss 0.8706 | val tinh khong nhan 0.0305 | co nhan 0.0304 | UAR 3 lop 0.963 
20   loss 0.8579 | val tinh khong nhan 0.0305 | co nhan 0.0304 | UAR 3 lop 0.965 
22   loss 0.8509 | val tinh khong nhan 0.0299 | co nhan 0.0298 | UAR 3 lop 0.970 <-- BEST
25   loss 0.8366 | val tinh khong nhan 0.0305 | co nhan 0.0304 | UAR 3 lop 0.964 
30   loss 0.8294 | val tinh khong nhan 0.0303 | co nhan 0.0302 | UAR 3 lop 0.966 
33   loss 0.8208 | val tinh khong nhan 0.0298 | co nhan 0.0297 | UAR 3 lop 0.959 

## Cell 11 - Danh gia tren test (chi clip neutral + 2 cam xuc da chon) + xuat cho demo

So: C4 (bang tra theo NHAN) · e2v+ -> nhan (chi 3 lop) -> bang tra · F 8 lop (Cell 6, chan con 3 lop) ·
**F2 chi audio** · F2 audio + nhan · tran. Them ket qua **tung cam xuc**.

In [12]:
TI = np.where(np.isin(DTE['emo'], KEEP))[0]                  # clip test thuoc 3 lop
STE = subset(DTE, KEEP)
S2, P2, V2_, C2 = pred2(STE, False); S2L, _, _, _ = pred2(STE, True)
# emotion2vec+ chi duoc chon trong 3 lop
pk = DTE['probs8'][TI][:, KEEP]; cas_lab = np.array(KEEP)[pk.argmax(1)]
cas_conf = pk.max(1) / np.maximum(pk.sum(1), 1e-8)
cas_lvl = np.clip(np.round(1 + 2 * np.clip((cas_conf - 0.4) / 0.5, 0, 1)), 1, 3).astype(int)
# F 8 lop (Cell 6) nhung chan con 3 lop
p8 = P_F[TI].copy(); m_ = np.zeros(8, bool); m_[KEEP] = True; p8[:, ~m_] = 0; f8_lab = p8.argmax(1)


def full_cond(sub):
    c = np.zeros((len(DTE['emo']), 7), np.float32); c[TI] = sub; return c


t0 = time.time()
MOT2 = {'casc': c4_motion(full_cond(np.stack([emo_vec(e, l) for e, l in zip(cas_lab, cas_lvl)]))), 'F2': c4_motion(full_cond(C2))}
print('Chuyen dong C4: %.0f s' % (time.time() - t0))
CANDS2 = {
    'dung yen':                          (np.zeros_like(STE['Y']), None),
    'C4 Phase B (bang tra theo NHAN)':   (S_table[TI], MOT['lab']),
    'e2v+ 3 lop -> nhan -> bang tra':    (np.stack([nearest_off(e, l) for e, l in zip(cas_lab, cas_lvl)]), MOT2['casc']),
    'F 8 lop (Cell 6, chi audio)':       (S_F[TI], MOT['F']),
    'F2 3 lop (CHI AUDIO)':              (S2, MOT2['F2']),
    'F2 3 lop (audio + nhan)':           (S2L, MOT['lab']),
    'tran: TB cua chinh clip':           (STE['Y'], None),
}
res2 = {}
for name, (S, M) in CANDS2.items():
    acc_n = {}
    for r, ci in enumerate(TI):
        p, n = CLIP[ci]; g = RAW[p:p + n]
        pr = S[r][None] + (M[p:p + n] if M is not None else 0)
        e = np.linalg.norm((g - pr).reshape(n, 68, 2), axis=2)
        for key in ('all_emo' if DTE['emo'][ci] > 0 else 'neutral', EMO_NAMES[DTE['emo'][ci]]):
            q = acc_n.setdefault(key, [np.zeros(68), 0]); q[0] += e.sum(0); q[1] += n
    res2[name] = {k: v[0] / v[1] for k, v in acc_n.items()}
base2 = res2['dung yen']
cols = ['all_emo', 'neutral'] + list(KEEP_NAMES)
print('\n%% LMD toan mat tot hon "dung yen" (test, clip thuoc 3 lop) | trong ngoac: mieng')
print('  %-34s ' % '' + ' '.join('%-17s' % c.replace('all_emo', 'ca 2 cam xuc') for c in cols))
report2 = dict(keep=KEEP_NAMES, best_epoch=BEST2, rows={})
for name in CANDS2:
    cells_ = []
    for c in cols:
        g_ = 100 * (1 - res2[name][c].mean() / base2[c].mean()); gm = 100 * (1 - res2[name][c][48:].mean() / base2[c][48:].mean())
        cells_.append('%+6.1f (%+6.1f)' % (g_, gm)); report2['rows'].setdefault(name, {})[c] = dict(face=round(g_, 2), mouth=round(gm, 2))
    print('  %-34s ' % name + ' '.join('%-17s' % s for s in cells_))
ye = DTE['emo'][TI]
def fc(S):
    with torch.no_grad():
        return (CLF_EVAL(torch.tensor(S / ZERO_LMD, dtype=torch.float32).to(DEVICE)) + KMASK).argmax(1).cpu().numpy()


print('\nEmo-Acc 3 lop (bo phan loai landmark doan cam xuc tu bieu cam tinh):')
for name, (S, _) in CANDS2.items():
    pc = fc(S); report2['rows'][name]['emo_acc'] = float((pc == ye).mean())
    print('  %-34s acc %.3f | UAR %.3f' % (name, (pc == ye).mean(), uar_sub(ye, pc)))
f2 = P2.argmax(1)
report2['ser'] = dict(F2_uar=uar_sub(ye, f2), F8_uar=uar_sub(ye, f8_lab), e2v_uar=uar_sub(ye, cas_lab),
                      F2_recall={EMO_NAMES[k]: round(float((f2[ye == k] == k).mean()), 3) for k in KEEP},
                      e2v_recall={EMO_NAMES[k]: round(float((cas_lab[ye == k] == k).mean()), 3) for k in KEEP})
print('\nNhan cam xuc tu audio (3 lop): F2 UAR %.3f | F 8 lop (chan 3) %.3f | emotion2vec+ (chan 3) %.3f'
      % (report2['ser']['F2_uar'], report2['ser']['F8_uar'], report2['ser']['e2v_uar']))
print('  recall F2 :', report2['ser']['F2_recall']); print('  recall e2v:', report2['ser']['e2v_recall'])
R = report2['rows']
print('\nKET LUAN (ca 2 cam xuc, toan mat): F2 chi audio %+.1f%% | C4 can nhan %+.1f%% | e2v+ -> nhan %+.1f%% | neutral: F2 %+.1f%% vs C4 %+.1f%%'
      % (R['F2 3 lop (CHI AUDIO)']['all_emo']['face'], R['C4 Phase B (bang tra theo NHAN)']['all_emo']['face'],
         R['e2v+ 3 lop -> nhan -> bang tra']['all_emo']['face'], R['F2 3 lop (CHI AUDIO)']['neutral']['face'],
         R['C4 Phase B (bang tra theo NHAN)']['neutral']['face']))
(OUT2 / 'phaseF2_evaluation.json').write_text(json.dumps(report2, indent=1, ensure_ascii=False, default=float))

# ---- xuat cho demo ----
head = dict(np.load(E2V_DIR / 'e2v_head.npz')) if (E2V_DIR / 'e2v_head.npz').exists() else None
torch.save(dict(branch_state=br2.state_dict(), branch_cfg=br2.cfg, keep=KEEP, keep_names=KEEP_NAMES, zero_lmd=ZERO_LMD,
                chunk_sec=CHUNK_SEC, e2v_size=E2V_SIZE, e2v_head=head, c4_ckpt=str(C4_CKPT), motion_pts=MOTION_PTS,
                label_offsets={'%d_%d' % k: v for k, v in OFF.items()}, best_epoch=BEST2, report=report2, emo_names=EMO_NAMES),
           OUT / 'phaseF2_final.pth')
if CK2_LAST.exists():
    CK2_LAST.unlink()
print('Da luu:', OUT / 'phaseF2_final.pth', '|', OUT2 / 'phaseF2_evaluation.json')

Chuyen dong C4: 323 s

%% LMD toan mat tot hon "dung yen" (test, clip thuoc 3 lop) | trong ngoac: mieng
                                     ca 2 cam xuc      neutral           angry             happy            
  dung yen                             +0.0 (  +0.0)     +0.0 (  +0.0)     +0.0 (  +0.0)     +0.0 (  +0.0)  
  C4 Phase B (bang tra theo NHAN)     +20.8 ( +23.8)     +6.1 ( +10.1)    +28.4 ( +26.1)     +9.6 ( +20.9)  
  e2v+ 3 lop -> nhan -> bang tra      +15.2 ( +18.1)     +4.2 (  +7.8)    +23.2 ( +20.4)     +3.6 ( +15.2)  
  F 8 lop (Cell 6, chi audio)         +18.3 ( +22.0)     -7.1 (  -0.8)    +25.3 ( +24.2)     +8.1 ( +19.3)  
  F2 3 lop (CHI AUDIO)                +18.6 ( +22.1)     +3.4 (  +8.2)    +25.1 ( +23.7)     +9.1 ( +20.0)  
  F2 3 lop (audio + nhan)             +20.3 ( +23.4)     +4.9 (  +9.3)    +27.5 ( +25.3)     +9.7 ( +21.0)  
  tran: TB cua chinh clip             +61.0 ( +48.5)    +28.2 ( +18.3)    +65.7 ( +50.4)    +54.1 ( +46.2)  

Emo-Acc 3 lop (bo phan 